# 🧠 Cleaning brain signals: removing artifacts from EEG

Welcome! In this notebook we'll take a real brain recording and **clean it up**, the same way researchers do before they analyse their data.

**You don't need to know how to program.** Each grey box below is a small piece of code. To run it:
- click on the box, then press **Shift + Enter** (or the ▶ button).
- wait until the `[ ]` on its left shows a number: that means it's done.
- run the boxes **in order**, from top to bottom.

### What we'll do today
1. 👀 **Meet the data**: what does an EEG recording look like?
2. 🔍 **Spot the "artifacts"**: signals that do *not* come from the brain
3. 🎚️ **Filtering**: removing slow drifts and fast noise
4. 🔌 **A broken electrode**: how to repair it
5. 👁️ **Eye blinks**: the biggest artifact in EEG
6. 🍸 **ICA, the "cocktail party" trick**: separating blinks from brain activity
7. ✨ **Before vs. after**: did it work?

We use **[MNE-Python](https://mne.tools)**, a free and widely used toolbox for analysing brain signals.

## 0. Setup

The next two boxes install and load the tools we need. It may take about a minute.

In [ ]:
# Install MNE on Colab. We hide pip's messages, because Colab shows harmless
# "dependency conflicts" warnings about packages we don't use.
import subprocess, sys
print("Installing MNE...")
result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "mne"],
                        capture_output=True, text=True)
print("Done ✅" if result.returncode == 0 else result.stderr)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import mne
from mne.datasets import eegbci

mne.set_log_level("ERROR")                 # keep the output tidy
mne.viz.set_browser_backend("matplotlib")  # static pictures
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("data/raw")                 # where downloaded recordings are stored
RAW_DIR.mkdir(parents=True, exist_ok=True)

### Settings

The recording we use comes from a public dataset ([PhysioNet EEG Motor Movement/Imagery](https://physionet.org/content/eegmmidb/1.0.0/)): 109 volunteers wore an EEG cap while opening and closing their fists. Each volunteer has a number.

👉 **Later, you can change `SUBJECT` to look at another person** (any number from 1 to 109).

In [ ]:
SUBJECT = 1   # which volunteer (1 to 109)
RUN = 4       # which recording session (4 = a 2-minute task)

## 1. 👀 Meet the data

**EEG** (electroencephalography) measures tiny electrical signals from the brain using electrodes placed on the scalp. These signals are *very* small: a few millionths of a volt (**microvolts, µV**). For comparison, a AA battery gives 1.5 volts, which is about a million times more.

Let's load one recording.

In [ ]:
fname = eegbci.load_data(SUBJECT, [RUN], path=RAW_DIR, update_path=False)[0]
raw = mne.io.read_raw_edf(fname, preload=True)
eegbci.standardize(raw)             # tidy up the electrode names
raw.set_montage("standard_1005")    # tell MNE where each electrode sits on the head
raw.set_annotations(None)           # remove the task markers: we don't need them today

print(f"Number of electrodes : {len(raw.ch_names)}")
print(f"Recording length     : {raw.times[-1]:.0f} seconds")
print(f"Measurements/second  : {raw.info['sfreq']:.0f} (per electrode)")

That's 64 electrodes, each measuring the brain **160 times per second**.

Where are these electrodes? Each dot below is one electrode, seen from above (the nose points up). Their names follow a convention: **F** = frontal (forehead), **C** = central, **P** = parietal, **O** = occipital (back of the head), **T** = temporal (near the ears). **Fp** electrodes sit just above the eyes. Keep that in mind!

In [ ]:
raw.plot_sensors(show_names=True, sphere="eeglab");

## 2. 🔍 Look at the raw signal

Now let's look at the signal itself. Each line is one electrode, and time runs from left to right. We show a few electrodes, from the forehead (**top**) to the back of the head (**bottom**), for 10 seconds.

In [ ]:
# A few electrodes, from front to back
SHOW = ["Fp1", "Fp2", "AF3", "F3", "Fz", "C3", "Cz", "P3", "Pz", "O1", "Oz"]

# Find the blinks, and pick the 10 seconds with the most of them, so there is something to see
blinks = mne.preprocessing.find_eog_events(raw.copy().filter(1, 15), ch_name=["Fp1", "Fp2"])
BLINK_TIMES = blinks[:, 0] / raw.info["sfreq"] - raw.first_time
START = max(np.arange(0, raw.times[-1] - 10, 0.5),
            key=lambda s: np.sum((BLINK_TIMES > s + 1) & (BLINK_TIMES < s + 9)))


def show(recording, title, start=START, duration=10, channels=SHOW, spacing=150, mark_blinks=False):
    # Plot the same electrodes and the same time window, so pictures are easy to compare
    idx = [recording.ch_names.index(ch) for ch in channels]
    data, times = recording.get_data(picks=idx, tmin=start, tmax=start + duration, return_times=True)
    data = (data - data.mean(axis=1, keepdims=True)) * 1e6   # volts -> microvolts, centred
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(channels) + 1.2))
    for i, trace in enumerate(data):
        ax.plot(times, trace - i * spacing, color="black", lw=0.7)
    if mark_blinks:
        for t in BLINK_TIMES[(BLINK_TIMES > start) & (BLINK_TIMES < start + duration)]:
            ax.axvspan(t - 0.2, t + 0.2, color="tab:blue", alpha=0.15)
    ax.set_yticks(-np.arange(len(channels)) * spacing, channels)
    ax.set_ylim(-len(channels) * spacing, 1.5 * spacing)
    ax.set_xlim(times[0], times[-1])
    ax.set_xlabel("Time (seconds)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.spines[["top", "right", "left"]].set_visible(False)
    # A scale bar, so people can see how big 100 microvolts is
    ax.plot([times[-1]] * 2, [spacing * 0.3, spacing * 0.3 + 100], color="tab:red", lw=3, clip_on=False)
    ax.text(times[-1], spacing * 0.3 + 50, "  100 µV", color="tab:red", va="center")
    plt.show()


def compare(before, after, title, channel="Fp1", start=START, duration=10):
    # Draw one electrode before (red) and after (black) cleaning, on top of each other
    fig, ax = plt.subplots(figsize=(12, 3))
    for recording, color, label in [(before, "tab:red", "before"), (after, "black", "after")]:
        data, times = recording.get_data(picks=[channel], tmin=start, tmax=start + duration,
                                         return_times=True)
        ax.plot(times, data[0] * 1e6, color=color, lw=0.9, label=label)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel(f"{channel} (µV)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.legend(loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


show(raw, "Raw signal, straight from the amplifier")

Look at the **top lines (Fp1, Fp2)**: do you see the big bumps standing out from the rest? Let's highlight them:

In [ ]:
show(raw, "The same signal, with the blinks highlighted", mark_blinks=True)

Those are **eye blinks!** 👁️ They are much bigger than the brain activity, but they don't come from the brain: the eye works like a small battery, and each time the eyelid moves, it changes the electric field near the forehead.

Signals like this are called **artifacts**: things the electrodes pick up that **don't come from the brain**. Researchers need to remove them, otherwise they could mistake a blink for brain activity.

| Artifact | Where it comes from | What it looks like |
|---|---|---|
| 👁️ Eye blinks | the eye and eyelid | big, slow bumps at the forehead |
| 👀 Eye movements | looking left/right | step-like changes near the eyes |
| 😬 Muscles (jaw, neck) | clenching, chewing, tension | fast, fuzzy "hairy" activity |
| ❤️ Heartbeat | the heart's electrical activity | small regular spikes |
| 🔌 Power line | electrical sockets (50 or 60 Hz) | a constant fast buzz |
| 💧 Sweat, movement | skin and cables | slow wandering drifts |
| ⚡ Bad electrode | poor contact with the scalp | a flat or extremely noisy line |

💡 *Live demo idea: if you have an EEG headset, ask a volunteer to blink, clench their jaw, or move their eyes, and watch these patterns appear.*

## 3. 🎚️ Filtering: removing slow drifts and fast noise

Brain signals are made of **rhythms**, slow and fast waves mixed together, like the instruments in a song. The speed of a wave is called its **frequency** and is measured in **Hertz (Hz)**: 10 Hz means 10 waves per second.

The plot below shows **how much of each frequency** is in our recording (a bit like the equalizer display on a music player).

In [ ]:
raw.compute_psd(fmax=80).plot(average=True, amplitude=False);

Most of the interesting brain activity lives roughly **between 1 and 40 Hz**:
- **below 1 Hz** there are mostly slow drifts (sweat, small movements of the cables)
- **above 40 Hz** there is mostly muscle activity and electrical noise
- 🔌 see the **sharp spike at 60 Hz**? That's the **power line**! The electricity in the wall sockets oscillates 60 times per second (this data was recorded in the USA; in Europe it's 50 Hz), and the electrodes pick it up like an antenna.

A **filter** works like the equalizer on your stereo: we turn down the frequencies we don't want and keep the ones we do. Here we keep **1 to 40 Hz**.

In [ ]:
filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)

filtered.compute_psd(fmax=80).plot(average=True, amplitude=False);

Everything above 40 Hz, including the 60 Hz power-line spike, has been pushed down. What does the signal look like now?

In [ ]:
show(raw, "BEFORE filtering")
show(filtered, "AFTER filtering (1-40 Hz)")

The lines are **smoother and more stable**. But look: **the blinks are still there!** 😕

That's because blinks are slow waves, in the *same* frequency range as brain activity. A filter can't separate them. We'll need a smarter trick (section 5). But first, a quick detour.

## 4. 🔌 A broken electrode

Sometimes an electrode loses contact with the scalp (dry gel, hair in the way, a loose cable). Then it records mostly noise.

Our recording is quite clean, so **we'll break one on purpose** 😈: we add a lot of noise to the electrode **Cz** (the top of the head).

In [ ]:
broken = filtered.copy()
rng = np.random.default_rng(0)
cz = broken.ch_names.index("Cz")
broken._data[cz] += rng.normal(0, 60e-6, broken.n_times)   # add strong noise to Cz

show(broken, "Cz has a bad contact")

**Cz** is clearly broken. What can we do?

1. Tell MNE that this electrode is **bad**.
2. **Interpolate** it: estimate what it should have recorded from its **neighbours**. It's like guessing the temperature in a town from the temperatures of the towns around it.

In [ ]:
broken.info["bads"] = ["Cz"]                   # 1. mark it as bad
repaired = broken.copy().interpolate_bads()    # 2. rebuild it from its neighbours

show(repaired, "Cz rebuilt from its neighbours")

Cz looks like a normal electrode again. 🎉

(We'll go on with the `filtered` recording, which never had this problem.)

## 5. 👁️ Eye blinks: the biggest artifact

Let's study the blinks more closely. MNE can **find every blink** automatically by looking for the big bumps on the forehead electrodes, and then **average them together** to show what a "typical blink" looks like.

(This dataset has no electrode dedicated to the eyes, so we use **Fp1** and **Fp2**, the electrodes just above the eyes.)

In [ ]:
blink_epochs = mne.preprocessing.create_eog_epochs(filtered, ch_name=["Fp1", "Fp2"],
                                                    baseline=(-0.5, -0.2))
print(f"Number of blinks found: {len(blink_epochs)}")

average_blink = blink_epochs.average()
average_blink.plot_joint(title="The average blink", times=[0.0], ts_args=dict(gfp=False));

How to read this picture:
- **the lines** are the electrodes. At time **0** the blink happens, and the signal gets very large.
- **the head map** shows *where* on the head the blink is strongest. **Red = strong**. It's all at the **front**, near the eyes, and fades towards the back.

**This "shape on the head" is the key to removing the blinks**, as we'll see next.

## 6. 🍸 ICA: the "cocktail party" trick

Imagine you're at a party 🎉 with **several people talking at the same time**, and you've placed **several microphones** around the room. Each microphone records a **mix** of all the voices, louder for the people standing close to it.

There's a mathematical method called **Independent Component Analysis (ICA)** that can take all these recordings and **separate the individual voices** again.

EEG is exactly the same situation:
- 🗣️ the **"voices"** are the different sources of activity: different brain areas... and the **eyes**, the **muscles**, the **heart**!
- 🎤 the **"microphones"** are our 64 electrodes, each recording a mix of everything

So the plan is:
1. **Separate** the signal into its "voices" (called **components**)
2. **Find the voice that belongs to the blinks**
3. **Mute it**, and put everything else back together

```
 electrodes (mixed)          components (separated)          electrodes (clean)
  Fp1  ~~~^~~~^~~                 brain  ~~~~~~~~~                  Fp1  ~~~~~~~~~
  Fz   ~~~^~~~^~~   ── ICA ──▶    brain  ~~~~~~~~~   ── mute ──▶    Fz   ~~~~~~~~~
  Oz   ~~~~~~~~~~                 blink  ___^___^_      👁️          Oz   ~~~~~~~~~
```

Let's separate the components. It only takes a few seconds.

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter="auto")
ica.fit(filtered)
print("Done! The signal was separated into", ica.n_components_, "components.")

We separated the signal into **20 components**. Here is the **"shape on the head"** of each one.

🕵️ **Your turn: which component looks like the blink?** Remember the average blink from section 5: strong at the front, near the eyes.

In [ ]:
ica.plot_components(sphere="eeglab");

Did you find it? Let's check with MNE: it compares each component with the forehead electrodes and gives a **score** (close to 1 or -1 = very similar to the blinks, close to 0 = unrelated).

In [ ]:
blink_components, scores = ica.find_bads_eog(filtered, ch_name=["Fp1", "Fp2"])
print("Component(s) that look like blinks:", [int(c) for c in blink_components])

ica.plot_scores(scores, exclude=blink_components, title="How similar is each component to the blinks?");

Let's also look at the **time course** of a few components. The blink component should show a bump every time the person blinks, while the others look like ongoing brain activity.

In [ ]:
ica.plot_sources(filtered, picks=range(5), start=START, stop=START + 10,
                 show_scrollbars=False);

**ICA000** has a sharp dip at each blink, while the others look like ongoing activity. (The blinks point *down* here instead of up: that's normal, ICA doesn't care about up or down.)

## 7. ✨ Remove the blinks and compare

Now we **mute** the blink component, and rebuild the electrode signals from all the others.

In [ ]:
ica.exclude = blink_components
cleaned = ica.apply(filtered.copy())

show(filtered, "BEFORE: filtered, blinks still there", mark_blinks=True)
show(cleaned, "AFTER: blink component removed", mark_blinks=True)

The blinks are gone from Fp1 and Fp2, and **the rest of the signal is still there**. 🎉

Here is the forehead electrode **Fp1** on a single picture (**red = before, black = after**):

In [ ]:
compare(filtered, cleaned, "Fp1, before and after removing the blinks")

One last check: the **average blink**, before and after cleaning (same scale for both).

In [ ]:
cleaned_blink = mne.Epochs(cleaned, blink_epochs.events, tmin=-0.5, tmax=0.5,
                           baseline=(-0.5, -0.2)).average()

ylim = dict(eeg=[-150, 250])
average_blink.plot(ylim=ylim, titles="Average blink BEFORE cleaning", gfp=False)
cleaned_blink.plot(ylim=ylim, titles="Same moments AFTER cleaning", gfp=False);

At the moments when the person blinked, there is now **almost nothing left**. ✅

## 🎓 Summary

| Problem | Tool | Idea |
|---|---|---|
| Slow drifts, fast noise | **Filter** | keep only the "brain" frequencies (1-40 Hz) |
| A broken electrode | **Interpolation** | rebuild it from its neighbours |
| Eye blinks | **ICA** | separate the "voices", mute the blink one |

Cleaning is usually the **first step** of any EEG study. Only after this do researchers start asking questions about the brain itself.

## 🧪 Try it yourself

1. Go back to **Settings**, change `SUBJECT` to another number (e.g. `3` or `7`), and run everything again (**Runtime → Run all** on Colab, **Run → Run All Cells** in Jupyter). Do the blinks look the same for everyone?
2. In section 7, remove a **wrong** component instead, e.g. `ica.exclude = [5]`. What happens to the signal?
3. In section 4, "break" a different electrode, such as `"Oz"` or `"Fp1"`.

---
### 📁 For the organisers: using your own recordings

Put your files in the `data/raw/` folder, and replace the loading cell in section 1 with:

```python
raw = mne.io.read_raw(RAW_DIR / "my_recording.edf", preload=True)   # .edf, .bdf, .fif, .set, .vhdr ...
raw.set_montage("standard_1005")
```

If your system has a dedicated eye electrode (often called `EOG` or `VEOG`), use it instead of `["Fp1", "Fp2"]` in sections 5 and 6. To pre-compute ICA in advance, see `scripts/prepare_data.py`.